## 1. Preparar o ambiente e localizar os arquivos

Importamos as bibliotecas necessárias (`pandas` e `numpy`) e definimos os caminhos das pastas do projeto. Em seguida, listamos todos os arquivos CSV brutos da ANP (de 2018 a 2020) que serão processados.

In [1]:
import os
import glob
import numpy as np
import pandas as pd

# Caminhos do repositório
PASTA_BRUTOS = os.path.join("..", "dados", "sujo")
PASTA_LIMPOS = os.path.join("..", "dados", "limpo")
os.makedirs(PASTA_LIMPOS, exist_ok=True)

arquivos_csv = sorted(glob.glob(os.path.join(PASTA_BRUTOS, "ca-*.csv")))
print(f"Arquivos encontrados ({len(arquivos_csv)}):", [os.path.basename(a) for a in arquivos_csv])

Arquivos encontrados (6): ['ca-2018-01.csv', 'ca-2018-02.csv', 'ca-2019-01.csv', 'ca-2019-02.csv', 'ca-2020-01.csv', 'ca-2020-02.csv']


## 2. Ler e unificar as bases semestrais

Lemos os seis arquivos CSV como texto para evitar perda de zeros à esquerda (como em CEPs e CNPJs) e conferimos se todos possuem exatamente 16 colunas. Também criamos a coluna `arquivo_origem` para rastrear de qual semestre veio cada registro antes de juntar tudo em uma única tabela.

In [2]:
dfs = []

for caminho in arquivos_csv:
    nome_arquivo = os.path.basename(caminho)
    df_part = pd.read_csv(
        caminho,
        sep=";",
        dtype=str,
        encoding="utf-8-sig",
        quotechar='"'
    )
    assert df_part.shape[1] == 16, f"Colunas inválidas em {nome_arquivo}: {df_part.shape[1]}"
    df_part["arquivo_origem"] = nome_arquivo
    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=True)
print(f"Total de registros extraídos: {len(df):,}")
df.head(3)

Total de registros extraídos: 2,686,331


,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira,arquivo_origem
0,NE,BA,SALVADOR,PETROBRAS DISTRIBUIDORA S.A.,34.274.233/0015-08,RUA EDISTIO PONDE,474,NaN,STIEP,41770-395,GNV,02/01/2018,"2,37","1,7383",R$ / m³,PETROBRAS DISTRIBUIDORA S.A.,ca-2018-01.csv
1,NE,BA,SALVADOR,PETROBRAS DISTRIBUIDORA S.A.,34.274.233/0015-08,RUA EDISTIO PONDE,474,NaN,STIEP,41770-395,DIESEL S10,02/01/2018,"3,24","3,1366",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.,ca-2018-01.csv
2,NE,BA,SALVADOR,PETROBRAS DISTRIBUIDORA S.A.,34.274.233/0015-08,RUA EDISTIO PONDE,474,NaN,STIEP,41770-395,ETANOL,02/01/2018,"2,93","2,6194",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.,ca-2018-01.csv


## 3. Limpar espaços em branco e mapear valores nulos

Removemos espaços sobrando no início e no fim de todos os campos de texto e convertemos strings vazias em valores nulos (`NA`). Por fim, exibimos a contagem de nulos por coluna — o que já evidencia a alta ausência de dados em `Complemento` e `Valor de Compra`.

In [3]:
colunas_texto = [c for c in df.columns if c != "arquivo_origem"]

for col in colunas_texto:
    df[col] = df[col].str.strip().replace("", pd.NA)

print("Contagem de nulos após trim:")
print(df[colunas_texto].isna().sum())

Contagem de nulos após trim:
Regiao - Sigla             0
Estado - Sigla             0
Municipio                  0
Revenda                    0
CNPJ da Revenda            0
Nome da Rua                0
Numero Rua              2067
Complemento          1996986
Bairro                  9353
Cep                        0
Produto                    0
Data da Coleta             0
Valor de Venda             0
Valor de Compra      1761960
Unidade de Medida          0
Bandeira                   0
dtype: int64


## 4. Padronizar CNPJ, CEP e número do endereço

Retiramos pontos, traços e barras do CNPJ e do CEP para manter apenas os dígitos, validando se todos têm 14 e 8 caracteres, respectivamente. Também uniformizamos as variações de "sem número" (como `SN`, `S.N.` ou vazios) para o padrão `S/N`.

In [4]:
df["cnpj"] = df["CNPJ da Revenda"].str.replace(r"\D", "", regex=True)
df["cep"] = df["Cep"].str.replace(r"\D", "", regex=True)

regex_sn = r"^(?:S\s*/?\s*N\.?\s*[º°O]?|SNR|SN)$"
df["numero_rua"] = (
    df["Numero Rua"]
    .fillna("S/N")
    .str.upper()
    .str.replace(regex_sn, "S/N", regex=True)
)

assert (df["cnpj"].str.len() == 14).all(), "CNPJs fora do padrão de 14 dígitos!"
assert (df["cep"].str.len() == 8).all(), "CEPs fora do padrão de 8 dígitos!"
print("CNPJ, CEP e numero_rua validados com sucesso.")

CNPJ, CEP e numero_rua validados com sucesso.


## 5. Converter datas e preços

Transformamos a data da coleta para o formato de data real (`datetime`) e convertemos o `Valor de Venda` para número decimal (trocando vírgula por ponto), garantindo que não existam datas inválidas nem preços zerados ou negativos.

In [ ]:
df["data_coleta"] = pd.to_datetime(df["Data da Coleta"], format="%d/%m/%Y", errors="coerce")

df["valor_venda"] = pd.to_numeric(
    df["Valor de Venda"].str.replace(",", ".", regex=False),
    errors="coerce"
)

assert df["data_coleta"].isna().sum() == 0, "Existem datas inválidas!"
assert (df["valor_venda"] > 0).all(), "Existem valores de venda <= 0!"


## 6. Padronizar os nomes das bandeiras

Aplicamos uma tabela de equivalência (*de-para*) para unificar nomes diferentes que representam a mesma distribuidora (como `ALE COMBUSTÍVEIS` virando `ALESAT`), mantendo também o nome original para referência.

In [6]:
caminho_de_para = os.path.join("..", "dados", "de_para_bandeira.csv")

if os.path.exists(caminho_de_para):
    df_de_para = pd.read_csv(caminho_de_para, sep=";", dtype=str, encoding="utf-8-sig")
    mapa_bandeira = dict(zip(df_de_para["bandeira_original"], df_de_para["bandeira_padronizada"]))
else:
    mapa_bandeira = {
        "PETROX DISTRIBUIDORA": "PETROX",
        "ALE COMBUSTÍVEIS": "ALESAT",
    }

df["bandeira_original"] = df["Bandeira"]
df["bandeira_padronizada"] = df["Bandeira"].replace(mapa_bandeira)

print(
    f"Bandeiras distintas: {df['bandeira_original'].nunique()} originais -> "
    f"{df['bandeira_padronizada'].nunique()} padronizadas"
)

Bandeiras distintas: 73 originais -> 71 padronizadas


## 7. Detectar preços atípicos (outliers) e checar duplicatas

Identificamos valores de venda muito fora do padrão (`fl_outlier`) usando o intervalo interquartil (IQR) agrupado por produto e mês. Também verificamos se há registros duplicados para o mesmo posto, produto e data, e confirmamos que cada combustível possui apenas uma unidade de medida.

In [ ]:
df["ano_mes"] = df["data_coleta"].dt.to_period("M")
grupo_prod_mes = df.groupby(["Produto", "ano_mes"])["valor_venda"]
q1 = grupo_prod_mes.transform("quantile", 0.25)
q3 = grupo_prod_mes.transform("quantile", 0.75)
iqr = q3 - q1

df["fl_outlier"] = (
    (df["valor_venda"] < (q1 - 3 * iqr)) |
    (df["valor_venda"] > (q3 + 3 * iqr))
)

qtd_duplicatas = df.duplicated(subset=["cnpj", "Produto", "data_coleta"]).sum()
print(f"Duplicatas na chave (cnpj, Produto, data_coleta): {qtd_duplicatas}")
print(f"fl_outlier = True: {df['fl_outlier'].sum():,}")
print("\nUnidades distintas por Produto:\n", df.groupby("Produto")["Unidade de Medida"].nunique())

Duplicatas na chave (cnpj, Produto, data_coleta): 0
fl_compra_maior_venda = True: 1,549
fl_outlier = True: 4,075

Unidades distintas por Produto:
 Produto
DIESEL                1
DIESEL S10            1
ETANOL                1
GASOLINA              1
GASOLINA ADITIVADA    1
GNV                   1
Name: Unidade de Medida, dtype: int64


## 8. Construir as tabelas dimensão e a tabela fato

Estruturamos os dados no formato de Modelo Estrela criando as cinco tabelas dimensão (`dim_tempo`, `dim_posto`, `dim_localizacao`, `dim_produto` e `dim_bandeira`) com suas respectivas chaves substitutas (*surrogate keys*). Em seguida, cruzamos essas chaves com os registros limpos para montar a tabela central `fato_preco`, mantendo apenas as métricas e colunas relevantes para a análise.

In [ ]:
# dim_tempo
datas = pd.date_range("2018-01-01", "2020-12-31", freq="D")
meses_pt = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
    5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
    9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
}
dias_pt = {
    0: "Segunda-feira", 1: "Terça-feira", 2: "Quarta-feira",
    3: "Quinta-feira", 4: "Sexta-feira", 5: "Sábado", 6: "Domingo"
}

dim_tempo = pd.DataFrame({"data": datas})
dim_tempo["sk_tempo"] = dim_tempo["data"].dt.strftime("%Y%m%d").astype(int)
dim_tempo["dia"] = dim_tempo["data"].dt.day
dim_tempo["mes"] = dim_tempo["data"].dt.month
dim_tempo["nome_mes"] = dim_tempo["mes"].map(meses_pt)
dim_tempo["trimestre"] = dim_tempo["data"].dt.quarter
dim_tempo["semestre"] = np.where(dim_tempo["mes"] <= 6, 1, 2)
dim_tempo["ano"] = dim_tempo["data"].dt.year
dim_tempo["dia_semana"] = dim_tempo["data"].dt.dayofweek.map(dias_pt)

condicoes = [
    dim_tempo["data"] < "2018-05-21",
    (dim_tempo["data"] >= "2018-05-21") & (dim_tempo["data"] <= "2018-06-05"),
    (dim_tempo["data"] > "2018-06-05") & (dim_tempo["data"] < "2020-03-11"),
    dim_tempo["data"] >= "2020-03-11"
]
periodos = ["Pré-Greve 2018", "Greve Caminhoneiros 2018", "Normalidade 2018-2019", "Pandemia 2020"]
dim_tempo["periodo_historico"] = np.select(condicoes, periodos, default="Outro")

# dim_posto (cadastro mais recente por CNPJ)
dim_posto = (
    # Desempate por CNPJ: deixa a numeração da sk_posto determinística (igual ao ELT)
    df.sort_values(["data_coleta", "cnpj"], kind="stable")
    .drop_duplicates(subset=["cnpj"], keep="last")
    [["cnpj", "Revenda", "Nome da Rua", "numero_rua", "Bairro", "cep"]]
    .rename(columns={
        "Revenda": "razao_social",
        "Nome da Rua": "rua",
        "Bairro": "bairro"
    })
    .reset_index(drop=True)
)
dim_posto.insert(0, "sk_posto", dim_posto.index + 1)

# dim_localizacao
dim_localizacao = (
    df[["Regiao - Sigla", "Estado - Sigla", "Municipio"]]
    .drop_duplicates()
    .rename(columns={"Regiao - Sigla": "regiao", "Estado - Sigla": "uf", "Municipio": "municipio"})
    .sort_values(["regiao", "uf", "municipio"])
    .reset_index(drop=True)
)
dim_localizacao.insert(0, "sk_localizacao", dim_localizacao.index + 1)

# dim_produto
dim_produto = (
    df[["Produto", "Unidade de Medida"]]
    .drop_duplicates()
    .rename(columns={"Produto": "produto", "Unidade de Medida": "unidade_medida"})
    .sort_values("produto")
    .reset_index(drop=True)
)
dim_produto.insert(0, "sk_produto", dim_produto.index + 1)

# dim_bandeira
dim_bandeira = (
    df[["bandeira_original", "bandeira_padronizada"]]
    .drop_duplicates()
    .sort_values(["bandeira_padronizada", "bandeira_original"])
    .reset_index(drop=True)
)
dim_bandeira["fl_branca"] = dim_bandeira["bandeira_padronizada"] == "BRANCA"
dim_bandeira.insert(0, "sk_bandeira", dim_bandeira.index + 1)

# fato_preco
fato_preco = (
    df.merge(dim_tempo[["data", "sk_tempo"]], left_on="data_coleta", right_on="data", how="left")
      .merge(dim_posto[["cnpj", "sk_posto"]], on="cnpj", how="left")
      .merge(
          dim_localizacao,
          left_on=["Regiao - Sigla", "Estado - Sigla", "Municipio"],
          right_on=["regiao", "uf", "municipio"],
          how="left"
      )
      .merge(
          dim_produto,
          left_on=["Produto", "Unidade de Medida"],
          right_on=["produto", "unidade_medida"],
          how="left"
      )
      .merge(
          dim_bandeira[["bandeira_original", "bandeira_padronizada", "sk_bandeira"]],
          on=["bandeira_original", "bandeira_padronizada"],
          how="left"
      )
      [[
          "sk_tempo", "sk_posto", "sk_produto", "sk_bandeira", "sk_localizacao",
          "valor_venda", "fl_outlier", "arquivo_origem"
      ]]
)

tabelas_dw = {
    "dim_tempo": dim_tempo,
    "dim_posto": dim_posto,
    "dim_localizacao": dim_localizacao,
    "dim_produto": dim_produto,
    "dim_bandeira": dim_bandeira,
    "fato_preco": fato_preco,
}

for nome, tabela in tabelas_dw.items():
    print(f"{nome}: {tabela.shape[0]:,} linhas x {tabela.shape[1]} colunas")

C:\Users\vinic\AppData\Local\Temp\ipykernel_6676\682758653.py:21: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dim_tempo["dia_semana"] = dim_tempo["data"].dt.dayofweek.map(dias_pt)


dim_tempo: 1,096 linhas x 10 colunas
dim_posto: 19,275 linhas x 8 colunas
dim_localizacao: 474 linhas x 4 colunas
dim_produto: 6 linhas x 3 colunas
dim_bandeira: 73 linhas x 4 colunas
fato_preco: 2,686,331 linhas x 11 colunas


## 9. Verificar o modelo antes da carga

Até aqui, as dimensões e a fato existem apenas como tabelas do Pandas. Antes de gravá-las
no PostgreSQL, conferimos o grão da fato e as chaves que ligarão cada preço às dimensões.
Se houver falha, a carga para e os dados problemáticos devem ser examinados.

In [ ]:
# Cada dimensão deve ter uma chave substituta preenchida e sem repetição.
chaves_dimensoes = {
    "dim_tempo": "sk_tempo",
    "dim_posto": "sk_posto",
    "dim_localizacao": "sk_localizacao",
    "dim_produto": "sk_produto",
    "dim_bandeira": "sk_bandeira",
}
for nome, chave in chaves_dimensoes.items():
    tabela = tabelas_dw[nome]
    if tabela[chave].isna().any() or tabela[chave].duplicated().any():
        raise ValueError(f"Chave {chave} inválida em {nome}")

# O grão adotado é um preço por data, posto e produto.
# A execução anterior encontrou zero duplicatas nessa combinação.
chave_natural_fato = ["sk_tempo", "sk_posto", "sk_produto"]
if fato_preco[chave_natural_fato].duplicated().any():
    raise ValueError("Há mais de um preço para a mesma data, posto e produto")

# Um LEFT JOIN sem correspondência pode gerar chaves estrangeiras vazias.
chaves_estrangeiras = [
    "sk_tempo", "sk_posto", "sk_produto", "sk_bandeira", "sk_localizacao"
]
if fato_preco[chaves_estrangeiras].isna().any().any():
    raise ValueError("A fato contém chaves sem dimensão correspondente")
if len(fato_preco) != len(df):
    raise ValueError("A quantidade de fatos difere da quantidade de registros lidos")

print(f"Modelo validado: {len(fato_preco):,} fatos e 5 dimensões")

## 10. Conectar ao PostgreSQL

Configure `PGHOST`, `PGDATABASE` e `PGUSER` no ambiente antes de executar esta célula.
`PGPORT` é opcional e usa 5432. A senha pode vir de `PGPASSWORD`; se ela não estiver
definida, o notebook pede a senha sem mostrá-la. Não salve credenciais no Git.

No Google Colab, o PostgreSQL precisa ser acessível a partir do runtime do Colab:
`localhost` lá não aponta para o computador pessoal.

In [ ]:
import getpass
from sqlalchemy import URL, create_engine, text

configuracoes_obrigatorias = ("PGHOST", "PGDATABASE", "PGUSER")
faltando = [nome for nome in configuracoes_obrigatorias if not os.getenv(nome)]
if faltando:
    raise RuntimeError(f"Configure as variáveis de ambiente: {', '.join(faltando)}")

senha = os.getenv("PGPASSWORD") or getpass.getpass("Senha do PostgreSQL: ")
url = URL.create(
    "postgresql+psycopg2",
    username=os.environ["PGUSER"],
    password=senha,
    host=os.environ["PGHOST"],
    port=int(os.getenv("PGPORT", "5432")),
    database=os.environ["PGDATABASE"],
)
engine = create_engine(url, pool_pre_ping=True)

# A consulta confirma a conexão sem imprimir a senha.
with engine.connect() as conexao:
    banco = conexao.execute(text("SELECT current_database()" )).scalar_one()
print(f"Conectado ao banco: {banco}")

## 11. Criar e carregar o Data Warehouse

O arquivo `sql/01_criar_dw_etl.sql` define as seis tabelas, chaves e relacionamentos.
Carregamos primeiro as dimensões e depois a fato, porque as chaves estrangeiras da fato
precisam encontrar suas dimensões. A transação confirma tudo ao final; se uma etapa
falhar, o PostgreSQL desfaz essa tentativa de carga.

Esta célula exige tabelas vazias. Isso evita duplicar os dados ao executar o notebook
pela segunda vez. Para refazer a carga, prepare um banco/esquema vazio de desenvolvimento.

In [ ]:
from pathlib import Path

# O notebook original lê os CSVs por ../dados/sujo, a partir de notebooks/.
# Usamos esse mesmo caminho para localizar o SQL sem alterar a extração existente.
raiz_projeto = Path(PASTA_BRUTOS).resolve().parents[1]
caminho_ddl = raiz_projeto / "sql" / "01_criar_dw_etl.sql"
if not caminho_ddl.is_file():
    raise FileNotFoundError(f"SQL de criação não encontrado: {caminho_ddl}")

ordem_carga = [
    "dim_tempo", "dim_posto", "dim_localizacao",
    "dim_produto", "dim_bandeira", "fato_preco",
]

# O arquivo contém apenas comandos CREATE separados por ponto e vírgula.
comandos_ddl = [
    comando.strip()
    for comando in caminho_ddl.read_text(encoding="utf-8").split(";")
    if comando.strip()
]

with engine.begin() as conexao:
    for comando in comandos_ddl:
        conexao.exec_driver_sql(comando)

    # O CREATE IF NOT EXISTS preserva tabelas já existentes. Exigimos que
    # estejam vazias para impedir uma segunda carga acidental.
    for nome in ordem_carga:
        existentes = conexao.execute(
            text(f"SELECT COUNT(*) FROM dw_etl.{nome}")
        ).scalar_one()
        if existentes:
            raise RuntimeError(f"dw_etl.{nome} já contém {existentes:,} linhas")

    for nome in ordem_carga:
        tabela = tabelas_dw[nome]
        tabela.to_sql(
            name=nome,
            con=conexao,
            schema="dw_etl",
            if_exists="append",
            index=False,
            chunksize=5000,
        )

        # Contamos no banco, pois o retorno de to_sql pode variar por driver.
        carregadas = conexao.execute(
            text(f"SELECT COUNT(*) FROM dw_etl.{nome}")
        ).scalar_one()
        if carregadas != len(tabela):
            raise RuntimeError(
                f"dw_etl.{nome}: {carregadas:,} no banco, {len(tabela):,} no Pandas"
            )
        print(f"dw_etl.{nome}: {carregadas:,} linhas")

print("Carga concluída e transação confirmada")

## 12. Conferir o resultado gravado

As consultas abaixo são executadas depois da confirmação da carga. Suas saídas são a
evidência de que o resultado do ETL está no banco de dados, como pede o enunciado.

In [ ]:
with engine.connect() as conexao:
    for nome in ordem_carga:
        total = conexao.execute(
            text(f"SELECT COUNT(*) FROM dw_etl.{nome}")
        ).scalar_one()
        print(f"{nome}: {total:,} linhas no PostgreSQL")

    # Uma amostra por arquivo comprova que os seis semestres chegaram à fato.
    por_arquivo = conexao.execute(text(
        "SELECT arquivo_origem, COUNT(*) AS total "
        "FROM dw_etl.fato_preco GROUP BY arquivo_origem ORDER BY arquivo_origem"
    )).all()
    for arquivo, total in por_arquivo:
        print(f"  {arquivo}: {total:,} fatos")

engine.dispose()